# PW01 · Source Event Shards

用途：在 Colab 冷启动环境中执行单个 positive_source shard。

边界：仅执行 PW01，不改动主链语义，不扩展到后续 stage。

## 1) 定义路径与参数

本单元定义 PW01 的输入参数，并固定脚本入口路径。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW01_Source_Event_Shards"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW01_Source_Event_Shards.py"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_demo"
SHARD_INDEX = 0
SHARD_COUNT = 2
FORCE_RERUN = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证 PW01 可以在空白 Colab 会话中直接冷启动。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
    },
)

## 3) 安装依赖并执行 attestation bootstrap

本单元和 PW00 对齐，确保依赖与 attestation 环境在冷启动后可直接使用。

In [ ]:
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(TRANSFORMERS_CACHE)

if Path("/usr/bin/apt-get").exists():
    subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"],
        check=False,
        capture_output=True,
    )

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

from scripts.notebook_runtime_common import ensure_attestation_env_bootstrap, load_yaml_mapping

CFG_OBJ = load_yaml_mapping(CONFIG_PATH)
ATTESTATION_BOOTSTRAP = ensure_attestation_env_bootstrap(
    CFG_OBJ,
    DRIVE_PROJECT_ROOT,
    allow_generate=True,
    allow_missing=False,
)
print_json("attestation_env_bootstrap", ATTESTATION_BOOTSTRAP)

## 4) 运行前 precheck

本单元检查 PW01 必要输入：family manifest、source event grid、source shard plan 与 shard 参数。

In [ ]:
FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "source_event_grid.jsonl"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW01 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source event grid 存在", SOURCE_EVENT_GRID_PATH.exists(), str(SOURCE_EVENT_GRID_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))

manifest_shard_count = None
if FAMILY_MANIFEST_PATH.exists():
    family_manifest_obj = json.loads(FAMILY_MANIFEST_PATH.read_text(encoding="utf-8"))
    source_parameters = family_manifest_obj.get("source_parameters", {})
    if isinstance(source_parameters, dict):
        manifest_shard_count = source_parameters.get("source_shard_count")

record_precheck(
    "shard 参数合法",
    isinstance(SHARD_COUNT, int) and SHARD_COUNT > 0 and isinstance(SHARD_INDEX, int) and 0 <= SHARD_INDEX < SHARD_COUNT,
    f"shard_index={SHARD_INDEX}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "manifest 与 SHARD_COUNT 一致",
    isinstance(manifest_shard_count, int) and manifest_shard_count == SHARD_COUNT,
    f"manifest_shard_count={manifest_shard_count}, shard_count={SHARD_COUNT}",
)
record_precheck("FORCE_RERUN 为 bool", isinstance(FORCE_RERUN, bool), str(FORCE_RERUN))
record_precheck(
    "attestation bootstrap 状态可用",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"},
    json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True),
)

print_json("pw01_precheck", {"items": PRECHECK_RESULTS})

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW01 precheck failed: {hard_fail}")

## 5) 执行 PW01 CLI

本单元执行单 shard 任务；若目标 shard 已完成，FORCE_RERUN=True 时会附加 --force-rerun。

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--shard-index",
    str(SHARD_INDEX),
    "--shard-count",
    str(SHARD_COUNT),
]
if FORCE_RERUN:
    COMMAND.append("--force-rerun")

PW01_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=build_repo_import_subprocess_env(repo_root=REPO_ROOT),
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW01_RESULT.returncode != 0:
    raise RuntimeError(
        "PW01 failed: "
        f"return_code={PW01_RESULT.returncode} stdout={PW01_RESULT.stdout} stderr={PW01_RESULT.stderr}"
    )

pw01_stdout_lines = [line.strip() for line in PW01_RESULT.stdout.splitlines() if line.strip()]
if not pw01_stdout_lines:
    raise RuntimeError("PW01 returned empty stdout, cannot parse summary JSON")

PW01_SUMMARY = json.loads(pw01_stdout_lines[-1])
print_json("pw01_summary", PW01_SUMMARY)

## 6) 回读 shard manifest

本单元验证 shard_manifest 已生成且状态为 completed。

In [ ]:
SHARD_MANIFEST_PATH = Path(PW01_SUMMARY["shard_manifest_path"])
if not SHARD_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"shard manifest missing: {SHARD_MANIFEST_PATH}")

SHARD_MANIFEST = json.loads(SHARD_MANIFEST_PATH.read_text(encoding="utf-8"))
print_json("pw01_shard_manifest", SHARD_MANIFEST)

if SHARD_MANIFEST.get("status") != "completed":
    raise RuntimeError(f"PW01 shard not completed: {SHARD_MANIFEST.get('status')}")

## 7) 如何扩展并行（显式说明）

扩展规则：
1. 先在 PW00 固定好 source_shard_count。
2. 多个执行器按 shard_index 分片并行执行 PW01。
3. 每个 shard 只写入 source_shards/positive/shard_xxxx，不会互相覆盖。
4. 仅在需要覆盖历史产物时使用 --force-rerun。

In [ ]:
parallel_plan = []
for shard_idx in range(SHARD_COUNT):
    shard_command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--drive-project-root",
        str(DRIVE_PROJECT_ROOT),
        "--family-id",
        FAMILY_ID,
        "--shard-index",
        str(shard_idx),
        "--shard-count",
        str(SHARD_COUNT),
    ]
    if FORCE_RERUN:
        shard_command.append("--force-rerun")
    parallel_plan.append(
        {
            "shard_index": shard_idx,
            "command": shard_command,
            "isolation_root": str(FAMILY_ROOT / "source_shards" / "positive" / f"shard_{shard_idx:04d}"),
        }
    )

print_json(
    "parallel_extension_plan",
    {
        "family_id": FAMILY_ID,
        "shard_count": SHARD_COUNT,
        "plan": parallel_plan,
    },
)